# Liquidation Strategy Walkthrough

This walkthrough follows one complete liquidation scenario for Liquidity Management Tools Calibration. It is written as a business walkthrough: first it tells the story of one representative redemption event, then it shows how to explore other sample configurations.

The walkthrough uses the existing sample data, loaders, and liquidation strategy engine. It does not add package logic, change methodology, or define new liquidation strategies.

## Setup

This setup cell loads shared helpers and pandas for readable review tables. The calculation work stays in the existing package engine.

In [ ]:
import pandas as pd
from _notebook_helpers import (
    ZERO,  # Decimal zero used for simple totals and comparisons
    cash_total,  # Calculates total cash in the loaded position list
    days,  # Formats day counts for display
    fund_positions,  # Selects positions for the walkthrough fund
    gross_sales,  # Adds up gross sale amounts from a liquidation result
    is_eligible,  # Checks whether a position can provide liquidity within the stress horizon
    load_sample_inputs,  # Loads all sample files through the existing project loaders
    money,  # Formats monetary values for display
    ordinary_portfolio_value,  # Adds portfolio market value excluding repo financing obligations
    print_profile,  # Prints a compact aligned key/value profile
    rate,  # Formats Decimal rates as percentages
    redemption_rows,  # Builds investor-class redemption rows for display
    run_liquidation_scenario,  # Calls the existing liquidation engine for a selected scenario
    scenario_from_selected_ids,  # Builds a scenario from the IDs chosen in the configuration cell
    yes_no,  # Formats booleans as yes/no labels
)
from _notebook_setup import SAMPLE_DATA_DIR, configure_display

configure_display()

## Load The Sample Inputs

The sample inputs are loaded as the same validated domain objects used elsewhere in the project. This keeps the walkthrough tied to the documented sample files instead of bypassing validation with ad hoc CSV parsing.

In [24]:
inputs = load_sample_inputs(SAMPLE_DATA_DIR)

sample_input_counts = [
    ("funds", len(inputs.funds)),
    ("positions", len(inputs.positions)),
    ("investor_classes", len(inputs.investor_classes)),
    ("redemption_scenarios", len(inputs.redemption_scenarios)),
    ("market_stresses", len(inputs.market_stresses)),
    ("liquidity_stresses", len(inputs.liquidity_stresses)),
    ("scenario_definitions", len(inputs.scenario_definitions)),
    ("lmt_parameters", len(inputs.lmt_parameters)),
    ("liquidation_strategies", len(inputs.liquidation_strategies)),
]

pd.DataFrame(sample_input_counts, columns=["dataset", "records"])

,dataset,records
0,funds,1
1,positions,9
2,investor_classes,5
3,redemption_scenarios,3
4,market_stresses,3
5,liquidity_stresses,3
6,scenario_definitions,4
7,lmt_parameters,2
8,liquidation_strategies,4


## Notebook Glue

Reusable display and scenario-assembly helpers live in `_notebook_helpers.py`: sample loading, value formatting, scenario assembly, and the call into the existing liquidation engine. The walkthrough below keeps only the selected scenario, the engine call, and result displays so the analysis reads top-to-bottom as a business story.

# Part 1 - Walkthrough

Part 1 follows one representative example from fund snapshot to final liquidation result. The example uses the existing `platform_outflow_hybrid` scenario because it shows how a fund can preserve its minimum cash buffer, use only part of available cash, and then sell eligible assets proportionally.

In [3]:
walkthrough_scenario_id = "platform_outflow_hybrid"
walkthrough_scenario = next(
    scenario
    for scenario in inputs.scenario_definitions
    if scenario.scenario_id == walkthrough_scenario_id
)
walkthrough = run_liquidation_scenario(inputs, walkthrough_scenario)
walkthrough_result = walkthrough.result

scenario_profile = {
    "scenario_id": walkthrough_scenario.scenario_id,
    "fund_id": walkthrough_scenario.fund_id,
    "redemption_scenario_id": walkthrough_scenario.redemption_scenario_id,
    "liquidity_stress_id": walkthrough_scenario.liquidity_stress_id,
    "liquidation_strategy_id": walkthrough_scenario.liquidation_strategy_id,
    "strategy_type": walkthrough.strategy.strategy_type.value,
    "lmt_parameter_set_id": walkthrough_scenario.lmt_parameter_set_id,
}

print_profile(scenario_profile)

             scenario_id:    platform_outflow_hybrid
                 fund_id:     lux_dynamic_allocation
  redemption_scenario_id:    severe_platform_outflow
     liquidity_stress_id:    reduced_equity_capacity
 liquidation_strategy_id: partial_cash_then_pro_rata
           strategy_type:                     hybrid
    lmt_parameter_set_id:        board_approved_base


## 1. Introduction

A redemption request creates a liquidity need: the fund must find usable cash without ignoring its liquidity constraints. The liquidation engine tests how the selected strategy meets that need within the stress horizon.

In this walkthrough, the reader will see how the engine preserves the minimum cash buffer, decides which positions are eligible, allocates sales, applies haircuts, and reports any remaining shortfall.

- Q: What is being tested?
- A: Whether the selected strategy can raise enough usable liquidity for the redemption request.

---

- Q: What does the engine return?
- A: Cash used, asset sales, post-haircut cash raised, dilution cost, shortfall, and remaining liquidity buffer.

---


- Q: What is outside this walkthrough?
- A: It does not decide whether an LMT should be activated and does not change methodology


## 2. The Fund

The selected fund is the synthetic Lux Dynamic Allocation Fund. Its NAV is the denominator for redemption rates, dilution rates, and the minimum cash buffer. Available cash is important, but the strategy is not allowed to treat all cash as freely usable because the LMT parameters preserve a minimum buffer.

In [4]:
fund = walkthrough.fund
fund_positions_for_walkthrough = fund_positions(inputs, fund)
fund_cash_total = cash_total(fund_positions_for_walkthrough)
portfolio_value = ordinary_portfolio_value(fund_positions_for_walkthrough)

fund_profile = {
    "fund_name": fund.fund_name,
    "fund_id": fund.fund_id,
    "as_of_date": str(fund.as_of_date),
    "base_currency": fund.base_currency,
    "nav": money(fund.nav),
    "available_cash": money(fund_cash_total),
    "ordinary_portfolio_value": money(portfolio_value),
    "dealing_frequency": fund.dealing_frequency,
}

print_profile(fund_profile)

                fund_name: Lux Dynamic Allocation Fund
                  fund_id:      lux_dynamic_allocation
               as_of_date:                  2026-06-30
            base_currency:                         EUR
                      nav:              100,000,000.00
           available_cash:               12,000,000.00
 ordinary_portfolio_value:              100,000,000.00
        dealing_frequency:                       daily


The portfolio is deliberately narrow for Version 1: cash, listed equities, listed ETFs, reverse repos, and repo financing exposure. Repo financing is shown for transparency, but it is not treated as an ordinary liquid asset for sale.

In [25]:
portfolio_table = pd.DataFrame(
    [
        {
            # "position_id": position.position_id,
            "instrument_name": position.instrument_name,
            "asset_group": position.asset_group.value,
            "market_value": money(position.market_value),
            "notional_amount": money(position.notional_amount),
            "base_haircut_rate": rate(position.base_haircut_rate),
            "base_liquidity_capacity_rate": rate(position.base_liquidity_capacity_rate),
            "settlement_days": days(position.settlement_days),
            "maturity_days": days(position.maturity_days),
        }
        for position in fund_positions_for_walkthrough
    ]
)

portfolio_table.style.set_properties(
    subset=["instrument_name"],
    **{"min-width": "300px", "white-space": "nowrap"},
)

,instrument_name,asset_group,market_value,notional_amount,base_haircut_rate,base_liquidity_capacity_rate,settlement_days,maturity_days
0,EUR Operating Cash,cash,"12,000,000.00",,0.00%,100.00%,0,
1,SAP SE Ordinary Shares,listed_equity,"15,000,000.00",,5.00%,20.00%,2,
2,ASML Holding NV Ordinary Shares,listed_equity,"12,000,000.00",,6.00%,18.00%,2,
3,LVMH Moet Hennessy Louis Vuitton SE Ordinary Shares,listed_equity,"8,000,000.00",,6.00%,18.00%,2,
4,iShares Core MSCI World UCITS ETF,listed_etf,"18,000,000.00",,4.00%,35.00%,2,
5,Xtrackers Euro Stoxx 50 UCITS ETF,listed_etf,"10,000,000.00",,4.00%,40.00%,2,
6,EUR Overnight Reverse Repo BNP Paribas Synthetic,reverse_repo,"15,000,000.00",,1.00%,100.00%,1,1
7,EUR One Week Reverse Repo Societe Generale Synthetic,reverse_repo,"10,000,000.00",,1.00%,90.00%,1,7
8,EUR Repo Financing Obligation Synthetic,repo_financing,,"5,000,000.00",0.00%,0.00%,1,


## 3. The Redemption Request

The redemption scenario converts investor-class stress assumptions into one redemption amount. The liquidation engine receives that total amount, but the class-level view explains where the cash demand comes from.

In [6]:
pd.DataFrame(
    [
        {
            "client_class": row["client_class"],
            "nav_share_rate": rate(row["nav_share_rate"]),
            "stress_redemption_rate": rate(row["stress_redemption_rate"]),
            "redemption_multiplier": str(row["redemption_multiplier"]),
            "notice_days": row["notice_days"],
            "settlement_days": row["settlement_days"],
            "redemption_amount": money(row["redemption_amount"]),
        }
        for row in redemption_rows(inputs, walkthrough_scenario)
    ]
)

,client_class,nav_share_rate,stress_redemption_rate,redemption_multiplier,notice_days,settlement_days,redemption_amount
0,retail,30.00%,8.00%,1.50,1,3,"3,600,000.00"
1,institutional,25.00%,12.00%,1.50,1,3,"4,500,000.00"
2,platform,25.00%,18.00%,1.50,0,3,"6,750,000.00"
3,fund_of_funds,15.00%,10.00%,1.50,2,4,"2,250,000.00"
4,seed_capital,5.00%,2.00%,1.50,30,30,"150,000.00"


The total redemption amount is the liquidity need the strategy must meet. A zero shortfall at the end means the selected strategy raised enough usable cash within the rules of the scenario.

In [7]:
redemption_profile = {
    "redemption_scenario_id": walkthrough.redemption.redemption_scenario_id,
    "total_redemption_amount": money(walkthrough.redemption_amount),
    "total_redemption_rate": rate(walkthrough.redemption_amount / fund.nav),
}

print(f"### {walkthrough.redemption.description} ###")
print()
print_profile(redemption_profile)

### Severe synthetic outflow led by platform distribution investors. ###

  redemption_scenario_id: severe_platform_outflow
 total_redemption_amount:           17,250,000.00
   total_redemption_rate:                  17.25%


## 4. Liquidity Constraints

The minimum cash buffer is taken from the selected LMT parameter set. The stress horizon is taken from the selected liquidity stress. Together, they constrain what liquidity can be used: cash below the buffer is preserved, and assets must settle or mature in time to be useful.

In [8]:
parameters = walkthrough.parameters
liquidity_stress = walkthrough.liquidity_stress
minimum_cash_buffer = fund.nav * parameters.minimum_buffer_rate
cash_above_buffer = max(fund_cash_total - minimum_cash_buffer, ZERO)

liq_constraints = {
    "lmt_parameter_set_id": parameters.parameter_set_id,
    "minimum_buffer_rate": rate(parameters.minimum_buffer_rate),
    "minimum_cash_buffer": money(minimum_cash_buffer),
    "available_cash": money(fund_cash_total),
    "cash_above_buffer": money(cash_above_buffer),
    "liquidity_stress_id": liquidity_stress.liquidity_stress_id,
    "liquidity_stress_multiplier": str(liquidity_stress.liquidity_stress_multiplier),
    "stress_horizon_days": liquidity_stress.stress_horizon_days,
}

print_profile(liq_constraints)

        lmt_parameter_set_id:     board_approved_base
         minimum_buffer_rate:                   5.00%
         minimum_cash_buffer:            5,000,000.00
              available_cash:           12,000,000.00
           cash_above_buffer:            7,000,000.00
         liquidity_stress_id: reduced_equity_capacity
 liquidity_stress_multiplier:                    2.00
         stress_horizon_days:                       5


## 5. Eligible Assets

Eligibility is based on asset group, positive market value, settlement timing, and reverse-repo maturity. Listed equities and ETFs must settle within the stress horizon. 
* Cash is handled separately
* Reverse repos must mature and settle within the horizon
* Repo financing exposures are financing transactions, not portfolio assets, and therefore are excluded from ordinary asset liquidation.

In [9]:
eligible_rows = []
for position in walkthrough.positions:
    available_capacity = None
    if position.stressed_market_value is not None:
        available_capacity = (
            position.stressed_market_value * position.stressed_liquidity_capacity_rate
        )
    eligible_rows.append(
        {
            "position_id": position.position_id,
            "asset_group": position.asset_group.value,
            "stressed_market_value": money(position.stressed_market_value),
            "stressed_haircut_rate": rate(position.stressed_haircut_rate),
            "stressed_liquidity_capacity_rate": rate(position.stressed_liquidity_capacity_rate),
            "available_capacity": money(available_capacity),
            "settlement_days": days(position.settlement_days),
            "maturity_days": days(position.maturity_days),
            "stress_horizon_days": liquidity_stress.stress_horizon_days,
            "eligible_for_liquidation": yes_no(
                is_eligible(position, liquidity_stress.stress_horizon_days)
            ),
        }
    )

pd.DataFrame(eligible_rows)[["position_id", "eligible_for_liquidation"]]

,position_id,eligible_for_liquidation
0,eur_operating_cash,no
1,sap_equity_position,yes
2,asml_equity_position,yes
3,lvmh_equity_position,yes
4,msci_world_etf_position,yes
5,euro_stoxx_etf_position,yes
6,overnight_reverse_repo_bnp,yes
7,one_week_reverse_repo_sg,no
8,eur_repo_financing_obligation,no


The one-week reverse repo is excluded in this example because its maturity plus settlement timing exceeds the stress horizon. Repo financing is excluded because it is a financing obligation rather than a liquid asset available for sale. 

Hence these are the assets eligible for liquidation.

In [10]:
pd.DataFrame([row for row in eligible_rows if row["eligible_for_liquidation"] == "yes"]).drop(
    ["eligible_for_liquidation"], axis=1
)

,position_id,asset_group,stressed_market_value,stressed_haircut_rate,stressed_liquidity_capacity_rate,available_capacity,settlement_days,maturity_days,stress_horizon_days
0,sap_equity_position,listed_equity,"15,000,000.00",10.00%,10.00%,"1,500,000.00",2,,5
1,asml_equity_position,listed_equity,"12,000,000.00",12.00%,9.00%,"1,080,000.00",2,,5
2,lvmh_equity_position,listed_equity,"8,000,000.00",12.00%,9.00%,"720,000.00",2,,5
3,msci_world_etf_position,listed_etf,"18,000,000.00",8.00%,17.50%,"3,150,000.00",2,,5
4,euro_stoxx_etf_position,listed_etf,"10,000,000.00",8.00%,20.00%,"2,000,000.00",2,,5
5,overnight_reverse_repo_bnp,reverse_repo,"15,000,000.00",2.00%,50.00%,"7,500,000.00",1,1,5


## 6. Liquidation Strategy

This scenario uses the `hybrid` strategy. It first uses a configured portion of cash above the minimum buffer, then allocates the remaining need across eligible non-cash assets on a pro-rata basis.

This strategy is useful when a fund wants to use some available cash but still avoid relying only on cash or only on asset sales.

In [11]:
strategy = walkthrough.strategy

liq_strategy = {
    "liquidation_strategy_id": strategy.liquidation_strategy_id,
    "strategy_type": strategy.strategy_type.value,
    # "description": strategy.description,
    "preserve_minimum_buffer": yes_no(strategy.preserve_minimum_buffer),
    "cash_buffer_use_rate": rate(strategy.cash_buffer_use_rate),
    "cash_available_above_buffer": money(cash_above_buffer),
    "cash_used_by_engine": money(walkthrough_result.cash_used),
}

print(f"## {strategy.description} ##")
print()
print_profile(liq_strategy)

## Uses part of cash above the configured buffer, then sells eligible non-cash assets proportionally. ##

     liquidation_strategy_id: partial_cash_then_pro_rata
               strategy_type:                     hybrid
     preserve_minimum_buffer:                        yes
        cash_buffer_use_rate:                     50.00%
 cash_available_above_buffer:               7,000,000.00
         cash_used_by_engine:               3,500,000.00


The gross sales below are selected by the engine. Gross sale amount is the value sold before haircut; post-haircut cash raised is the usable liquidity after the stressed haircut.

In [12]:
pd.DataFrame(
    [
        {
            "position_id": asset.position_id,
            "asset_group": asset.asset_group.value,
            "gross_sale_amount": money(asset.gross_sale_amount),
            "post_haircut_cash_raised": money(asset.post_haircut_cash_raised),
            "haircut_cost": money(asset.haircut_cost),
        }
        for asset in walkthrough_result.assets_liquidated
    ]
)

,position_id,asset_group,gross_sale_amount,post_haircut_cash_raised,haircut_cost
0,asml_equity_position,listed_equity,"1,080,000.00","950,400.00","129,600.00"
1,euro_stoxx_etf_position,listed_etf,"1,916,109.25","1,762,820.51","153,288.74"
2,lvmh_equity_position,listed_equity,"720,000.00","633,600.00","86,400.00"
3,msci_world_etf_position,listed_etf,"3,150,000.00","2,898,000.00","252,000.00"
4,overnight_reverse_repo_bnp,reverse_repo,"2,698,194.66","2,644,230.77","53,963.89"
5,sap_equity_position,listed_equity,"1,500,000.00","1,350,000.00","150,000.00"


## 7. Liquidation Result

The result brings the strategy together: cash used, gross asset sales, usable post-haircut cash, dilution cost, and any remaining shortfall. Dilution cost is the haircut leakage caused by selling assets under stressed liquidity assumptions.

In [13]:
walkthrough_gross_sales = gross_sales(walkthrough_result)
total_cash_raised = walkthrough_result.cash_used + walkthrough_result.total_post_haircut_cash_raised

liq_result = {
    "total_redemption_amount": money(walkthrough_result.total_redemption_amount),
    "cash_used": money(walkthrough_result.cash_used),
    "gross_sales": money(walkthrough_gross_sales),
    "post_haircut_cash_raised": money(walkthrough_result.total_post_haircut_cash_raised),
    "total_cash_raised": money(total_cash_raised),
    "dilution_cost": money(walkthrough_result.dilution_amount),
    "dilution_rate": rate(walkthrough_result.dilution_rate),
    "remaining_shortfall": money(walkthrough_result.shortfall),
    "remaining_liquid_buffer_rate": rate(walkthrough_result.remaining_liquid_buffer_rate),
    "minimum_cash_buffer_preserved": yes_no(walkthrough_result.minimum_cash_buffer_preserved),
}

print_profile(liq_result)

       total_redemption_amount: 17,250,000.00
                     cash_used:  3,500,000.00
                   gross_sales: 11,064,303.92
      post_haircut_cash_raised: 10,239,051.28
             total_cash_raised: 13,739,051.28
                 dilution_cost:    825,252.63
                 dilution_rate:         0.83%
           remaining_shortfall:  3,510,948.72
  remaining_liquid_buffer_rate:        13.28%
 minimum_cash_buffer_preserved:           yes


Allocation by asset group helps the reviewer see the portfolio-level behavior of the strategy. These are gross allocation amounts, so they should be interpreted alongside the haircut and post-haircut cash figures above.

In [14]:
alloc_class_breakdown = {
    asset_group.value: money(allocation)
    for asset_group, allocation in walkthrough_result.asset_group_allocations.items()
}
print_profile(alloc_class_breakdown)

          cash: 3,500,000.00
 listed_equity: 3,300,000.00
    listed_etf: 5,066,109.25
  reverse_repo: 2,698,194.66


## 8. Final Summary

In this example, the redemption is successfully met if remaining shortfall is zero. The reviewer should conclude that, under the selected synthetic assumptions, the hybrid strategy can raise enough usable liquidity while preserving the configured minimum cash buffer. The result should still be reviewed for dilution cost and remaining liquidity buffer pressure.

In [15]:
print(
    f"Reviewer_conclusion: "
    f"{
        'Redemption need is met within the scenario constraints.'
        if walkthrough_result.shortfall == ZERO
        else 'Redemption need is not fully met within the scenario constraints.'
    }"
)

print()

summary = {
    "scenario_id": walkthrough_scenario.scenario_id,
    "redemption_met": yes_no(walkthrough_result.shortfall == ZERO),
    "remaining_shortfall": money(walkthrough_result.shortfall),
    "minimum_cash_buffer_preserved": yes_no(walkthrough_result.minimum_cash_buffer_preserved),
    "dilution_cost": money(walkthrough_result.dilution_amount),
}

print_profile(summary)

Reviewer_conclusion: Redemption need is not fully met within the scenario constraints.

                   scenario_id: platform_outflow_hybrid
                redemption_met:                      no
           remaining_shortfall:            3,510,948.72
 minimum_cash_buffer_preserved:                     yes
                 dilution_cost:              825,252.63


# Part 2 - Exploring Different Scenarios

After the walkthrough, change the selected inputs to explore other validated sample scenarios. The tables below show the available funds, redemption scenarios, liquidity stresses, LMT parameter sets, and liquidation strategies discovered through the existing loaders.

## Available Funds

Funds are added to `data/sample/funds.csv`. The loader discovers them when the sample inputs are reloaded.

In [16]:
pd.DataFrame(
    [
        {
            # "fund_id": fund.fund_id,
            "fund_name": fund.fund_name,
            "as_of_date": str(fund.as_of_date),
            "base_currency": fund.base_currency,
            "nav": money(fund.nav),
            "dealing_frequency": fund.dealing_frequency,
        }
        for fund in inputs.funds
    ]
)

,fund_name,as_of_date,base_currency,nav,dealing_frequency
0,Lux Dynamic Allocation Fund,2026-06-30,EUR,"100,000,000.00",daily


## Available Redemption Scenarios

Redemption scenarios are added to `data/sample/redemption_scenarios.csv`. They are reusable liability-side assumptions and do not contain liquidation strategy choices.

In [26]:
pd.DataFrame(
    [
        {
            "redemption_scenario_id": scenario.redemption_scenario_id,
            # "name": scenario.name,
            "description": scenario.description,
            "redemption_multiplier": str(scenario.redemption_multiplier),
        }
        for scenario in inputs.redemption_scenarios
    ]
).style.set_properties(
    subset=["description"],
    **{"min-width": "300px", "white-space": "nowrap"},
)

,redemption_scenario_id,description,redemption_multiplier
0,moderate_redemption_pressure,Moderate synthetic redemption pressure across investor classes.,1.00
1,severe_platform_outflow,Severe synthetic outflow led by platform distribution investors.,1.50
2,extreme_broad_based_outflow,Extreme synthetic broad-based redemption pressure.,2.00


## Available Liquidity Stress Scenarios

Liquidity stresses are added to `data/sample/liquidity_stresses.json`. The stress horizon controls which assets can provide usable liquidity in time.

In [28]:
pd.DataFrame(
    [
        {
            "liquidity_stress_id": stress.liquidity_stress_id,
            # "name": stress.name,
            "description": stress.description,
            "liquidity_stress_multiplier": str(stress.liquidity_stress_multiplier),
            "stress_horizon_days": stress.stress_horizon_days,
        }
        for stress in inputs.liquidity_stresses
    ]
).style.set_properties(
    subset=["description"],
    **{"min-width": "300px", "white-space": "nowrap"},
)

,liquidity_stress_id,description,liquidity_stress_multiplier,stress_horizon_days
0,normal_liquidity_capacity,Normal synthetic liquidity capacity assumptions.,1.00,5
1,reduced_equity_capacity,Reduced synthetic liquidity capacity for listed instruments.,2.00,5
2,severe_liquidity_squeeze,Severe synthetic liquidity squeeze with shorter stress horizon.,3.00,3


## Available LMT Parameter Sets

LMT parameter sets are added to `data/sample/lmt_parameters.csv`. The minimum buffer rate from the selected set determines the cash buffer preserved by the liquidation strategy.

In [30]:
pd.DataFrame(
    [
        {
            "fund_id": parameters.fund_id,
            "as_of_date": str(parameters.as_of_date),
            "parameter_set_id": parameters.parameter_set_id,
            "swing_threshold_rate": rate(parameters.swing_threshold_rate),
            "max_swing_factor_rate": rate(parameters.max_swing_factor_rate),
            "gate_threshold_rate": rate(parameters.gate_threshold_rate),
            "minimum_buffer_rate": rate(parameters.minimum_buffer_rate),
        }
        for parameters in inputs.lmt_parameters
    ]
).style.set_properties(
    subset=["as_of_date"],
    **{"min-width": "12px", "white-space": "nowrap"},
)

,fund_id,as_of_date,parameter_set_id,swing_threshold_rate,max_swing_factor_rate,gate_threshold_rate,minimum_buffer_rate
0,lux_dynamic_allocation,2026-06-30,board_approved_base,1.50%,3.00%,10.00%,5.00%
1,lux_dynamic_allocation,2026-06-30,conservative_liquidity_buffer,1.00%,2.50%,8.00%,8.00%


## Available Liquidation Strategies

Liquidation strategies are configured in `data/sample/liquidation_strategies.json`. They are discovered through the existing JSON loader.

`most_liquid_first` is typically appropriate when a reviewer wants to test a cash-first and most-liquid-asset-first response while still preserving the minimum cash buffer.

`pro_rata` is typically appropriate when a reviewer wants sales to preserve the eligible portfolio profile rather than concentrating sales in the most liquid holdings.

`hybrid` is typically appropriate when a reviewer wants to use part of excess cash first, then spread remaining sales across eligible non-cash assets.

`custom_weights` is typically appropriate when a reviewer wants a documented allocation mix by asset group, using weights configured in JSON rather than hardcoded in the walkthrough.

In [31]:
pd.DataFrame(
    [
        {
            "liquidation_strategy_id": strategy.liquidation_strategy_id,
            "strategy_type": strategy.strategy_type.value,
            "description": strategy.description,
            # "preserve_minimum_buffer": yes_no(strategy.preserve_minimum_buffer),
            "cash_buffer_use_rate": rate(strategy.cash_buffer_use_rate),
            "weights": (
                ", ".join(
                    f"{asset_group.value}: {rate(weight)}"
                    for asset_group, weight in strategy.weights.items()
                )
                if strategy.weights
                else ""
            ),
        }
        for strategy in inputs.liquidation_strategies
    ]
).style.set_properties(
    subset=["description", "weights"],
    **{"min-width": "12px", "white-space": "nowrap"},
)

,liquidation_strategy_id,strategy_type,description,cash_buffer_use_rate,weights
0,cash_then_liquid_assets,most_liquid_first,"Uses cash above the configured minimum buffer first, then eligible liquid assets.",,
1,portfolio_profile_pro_rata,pro_rata,Sells eligible non-cash assets proportionally to preserve the portfolio liquidity profile.,,
2,partial_cash_then_pro_rata,hybrid,"Uses part of cash above the configured buffer, then sells eligible non-cash assets proportionally.",50.00%,
3,balanced_custom_weights,custom_weights,"Allocates liquidation needs across ETFs, listed equities, and reverse repos using configured weights.",50.00%,"listed_etf: 45.00%, listed_equity: 35.00%, reverse_repo: 20.00%"


## Configuration Cell

Change the selected IDs in this single cell, then rerun the cells below it. The IDs must exist in the available-options tables above.

In [21]:
selected_fund_id = "lux_dynamic_allocation"
selected_redemption_scenario_id = "severe_platform_outflow"
selected_liquidity_stress_id = "reduced_equity_capacity"
selected_lmt_parameter_id = "board_approved_base"
selected_strategy_id = "partial_cash_then_pro_rata"

## Selected Configuration Result

This cell assembles the selected sample records and calls the existing liquidation strategy engine. Use the result as an educational comparison against the walkthrough, not as a production report.

In [22]:
selected_scenario = scenario_from_selected_ids(
    inputs,
    fund_id=selected_fund_id,
    redemption_scenario_id=selected_redemption_scenario_id,
    liquidity_stress_id=selected_liquidity_stress_id,
    lmt_parameter_id=selected_lmt_parameter_id,
    strategy_id=selected_strategy_id,
)
selected_run = run_liquidation_scenario(inputs, selected_scenario)
selected_result = selected_run.result
selected_gross_sales = gross_sales(selected_result)

selected_run = {
    "fund_id": selected_fund_id,
    "redemption_scenario_id": selected_redemption_scenario_id,
    "liquidity_stress_id": selected_liquidity_stress_id,
    "lmt_parameter_id": selected_lmt_parameter_id,
    "strategy_id": selected_strategy_id,
    "strategy_type": selected_run.strategy.strategy_type.value,
    "total_redemption_amount": money(selected_result.total_redemption_amount),
    "cash_used": money(selected_result.cash_used),
    "gross_sales": money(selected_gross_sales),
    "post_haircut_cash_raised": money(selected_result.total_post_haircut_cash_raised),
    "dilution_cost": money(selected_result.dilution_amount),
    "dilution_rate": rate(selected_result.dilution_rate),
    "remaining_shortfall": money(selected_result.shortfall),
    "remaining_liquid_buffer_rate": rate(selected_result.remaining_liquid_buffer_rate),
    "minimum_cash_buffer_preserved": yes_no(selected_result.minimum_cash_buffer_preserved),
}
print_profile(selected_run)

                       fund_id:     lux_dynamic_allocation
        redemption_scenario_id:    severe_platform_outflow
           liquidity_stress_id:    reduced_equity_capacity
              lmt_parameter_id:        board_approved_base
                   strategy_id: partial_cash_then_pro_rata
                 strategy_type:                     hybrid
       total_redemption_amount:              17,250,000.00
                     cash_used:               3,500,000.00
                   gross_sales:              11,064,303.92
      post_haircut_cash_raised:              10,239,051.28
                 dilution_cost:                 825,252.63
                 dilution_rate:                      0.83%
           remaining_shortfall:               3,510,948.72
  remaining_liquid_buffer_rate:                     13.28%
 minimum_cash_buffer_preserved:                        yes


The position-level allocation below shows which assets the selected strategy uses. When comparing strategies, focus on the pattern of gross sales, haircut cost, and shortfall rather than only the total cash raised.

In [23]:
pd.DataFrame(
    [
        {
            "position_id": asset.position_id,
            "asset_group": asset.asset_group.value,
            "gross_sale_amount": money(asset.gross_sale_amount),
            "post_haircut_cash_raised": money(asset.post_haircut_cash_raised),
            "haircut_cost": money(asset.haircut_cost),
        }
        for asset in selected_result.assets_liquidated
    ]
)

,position_id,asset_group,gross_sale_amount,post_haircut_cash_raised,haircut_cost
0,asml_equity_position,listed_equity,"1,080,000.00","950,400.00","129,600.00"
1,euro_stoxx_etf_position,listed_etf,"1,916,109.25","1,762,820.51","153,288.74"
2,lvmh_equity_position,listed_equity,"720,000.00","633,600.00","86,400.00"
3,msci_world_etf_position,listed_etf,"3,150,000.00","2,898,000.00","252,000.00"
4,overnight_reverse_repo_bnp,reverse_repo,"2,698,194.66","2,644,230.77","53,963.89"
5,sap_equity_position,listed_equity,"1,500,000.00","1,350,000.00","150,000.00"


## Extending The Examples

Add new funds in `data/sample/funds.csv`, with matching positions in `data/sample/positions.csv`, investor classes in `data/sample/investor_classes.csv`, and LMT parameters in `data/sample/lmt_parameters.csv`.

Add new redemption scenarios in `data/sample/redemption_scenarios.csv` and new liquidity stress scenarios in `data/sample/liquidity_stresses.json`.

Add new liquidation strategy configurations in `data/sample/liquidation_strategies.json`, using only the strategy types already supported by the engine.

After changing the sample files, rerun the walkthrough from the top. The existing loaders validate and discover the available records automatically, and the option tables update from those loaded objects.